In [ ]:
# Cell 1 — download a small GGUF model. (Previously also downloaded a prebuilt CUDA
# binary of llama.cpp's own server, but that asset no longer exists in current releases —
# llama.cpp only ships prebuilt CUDA binaries for Windows now, confirmed via the GitHub
# releases API. Removed; Cell 2 uses a prebuilt llama-cpp-python CUDA wheel instead.)
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(
    repo_id='Qwen/Qwen2.5-Coder-7B-Instruct-GGUF',
    filename='qwen2.5-coder-7b-instruct-q8_0.gguf'
)
print('MODEL_PATH:', model_path)


In [ ]:
# Cell 2 — try a prebuilt CUDA wheel first (seconds, not the ~20-30 min a from-source
# build takes), falling back to compiling from source only if that fails. mlock disabled
# (matches the exact load_mode terminology from an earlier hang); polling health-check
# loop instead of a blind sleep so failures surface in ~1 minute instead of hours.
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.returncode, result.stdout + result.stderr

print('--- CUDA toolchain check ---')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo 'NVIDIA_SMI_FAILED'
!nvcc --version || echo 'NVCC_NOT_FOUND'

print('--- attempting prebuilt CUDA wheel (fast path, no compilation) ---')
rc, out = run('pip install -q llama-cpp-python[server] --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --force-reinstall --no-cache-dir')
print(out[-3000:])
print('pip exit code:', rc)

import importlib
prebuilt_ok = False
try:
    importlib.import_module('llama_cpp')
    prebuilt_ok = True
    print('IMPORT_OK: prebuilt CUDA wheel installed successfully')
except ImportError as e:
    print(f'PREBUILT_WHEEL_FAILED: {e}')

if not prebuilt_ok:
    print('--- prebuilt wheel did not work, falling back to building from source (slow, ~20-30 min) ---')
    import os
    os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
    os.environ['FORCE_CMAKE'] = '1'
    os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] = '4'
    !pip install -q --no-cache-dir --force-reinstall llama-cpp-python[server] > build.log 2>&1
    with open('build.log') as f:
        lines = f.readlines()
    print(f'build.log has {len(lines)} lines')
    error_lines = [i for i, l in enumerate(lines) if "error" in l.lower()]
    if error_lines:
        start = max(0, error_lines[0] - 20)
        print(''.join(lines[start:error_lines[0] + 10]))
    else:
        print(''.join(lines[-60:]))
    try:
        importlib.import_module('llama_cpp')
        prebuilt_ok = True
        print('IMPORT_OK: source build succeeded')
    except ImportError as e:
        print(f'IMPORT_FAILED: {e}')

import subprocess as sp, time, requests

server_proc = sp.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', model_path,
    '--host', '0.0.0.0',
    '--port', '8000',
    '--n_gpu_layers', '-1',
    '--use_mlock', 'false',
])

print('SERVER_STARTING pid=', server_proc.pid)
ready = False
for i in range(30):
    if server_proc.poll() is not None:
        print('SERVER_PROCESS_EXITED code=', server_proc.returncode)
        break
    try:
        resp = requests.get('http://localhost:8000/v1/models', timeout=3)
        if resp.status_code == 200:
            print(f'SERVER_READY after {i * 10}s')
            ready = True
            break
    except Exception:
        pass
    print(f'  ...waiting for server ({i * 10}s elapsed)')
    time.sleep(10)

if not ready:
    print('SERVER_NOT_READY after 300s -- check output above for errors')


In [ ]:
# Cell 3 — start cloudflared tunnel, print the public URL.
# Only keep the session alive forever (the tunnel keep-alive loop) if the server
# actually started in cell 2. Otherwise a failed run would idle for the full 9-hour
# batch session cap doing nothing useful, occupying one of the 2 concurrent GPU
# session slots -- this is exactly what happened with the v2/v3 pushes.
if not globals().get('ready', False):
    print('SKIPPING_TUNNEL: server was not ready after cell 2, exiting so this session frees up promptly')
else:
    !curl -L -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared
    import subprocess, time, re
    tunnel_proc = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    url = None
    for _ in range(60):
        line = tunnel_proc.stdout.readline()
        print(line, end='')
        match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break
    print()
    print('TUNNEL_URL:', url)
    print('Keep this cell running -- closing it kills the tunnel.')
    while True:
        time.sleep(60)
